# 🏡 FHBG Eligibility Bot - Interactive Demo

This Jupyter notebook demonstrates the First Home Buyer Grant Eligibility Bot.

**Features:**
- Agent collaboration demonstration
- Real-time eligibility checking
- Step-by-step conversation flow
- Report generation

In [ ]:
# Add src to path
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

%load_ext autoreload
%autoreload 2

# Import our agents
from src.agents.rule_scraper import RuleScraper
from src.agents.rule_interpreter import RuleInterpreter
from src.agents.conversation_manager import ConversationManager
from src.agents.reporter import ReportGenerator

## 🕷️ Agent 1: Rule Scraper

Fetches (or loads cached) FHBG eligibility rules from state government sources.

In [ ]:
# Initialize scraper
scraper = RuleScraper(data_path="src/data/", cache_ttl=3600)

# Scrape NSW rules
print("Scraping NSW rules...")
nsw_rules = scraper.scrape_state_rules("NSW", force_refresh=True)

print(f"✅ Loaded rules for: {nsw_rules['state']}")
print(f"   Grant: {nsw_rules['grant_name']}")
print(f"   Amount: ${nsw_rules['rules']['grant_amount']:,}")
print(f"   Income cap: ${nsw_rules['rules']['income_cap']:,}")
print(f"   Property price cap: ${nsw_rules['rules']['property_price_cap']:,}")

# Show all keys
print("\nFull rules structure:")
import json
print(json.dumps(nsw_rules, indent=2)[:1500], "...")

## ✅ Agent 2: Rule Interpreter

Validates user data against the scraped rules.

In [ ]:
# Initialize interpreter with loaded rules
interpreter = RuleInterpreter(nsw_rules)

# Test with sample eligible user
eligible_user = {
    "state": "NSW",
    "income": 80000,
    "property_price": 700000,
    "first_home_buyer": True,
    "citizenship_status": "australian citizen",
    "will_reside": True,
    "property_is_new": False,
}

print("=== Testing ELIGIBLE user ===")
report = interpreter.validate_eligibility(eligible_user)

print(f"Status: {report.status.value.upper()}")
print(f"Grant: ${report.grant_amount:,}")
print(f"Passed rules: {len(report.passed_rules)}")
print(f"Failed rules: {len(report.failed_rules)}")
print("\nNext steps:")
for step in report.next_steps:
    print(f"  - {step}")

In [ ]:
# Test with NOT eligible user (income too high)
ineligible_user = {
    "state": "NSW",
    "income": 200000,  # Above cap
    "property_price": 700000,
    "first_home_buyer": True,
    "citizenship_status": "citizen",
    "will_reside": True,
}

print("=== Testing NOT ELIGIBLE user (high income) ===")
report2 = interpreter.validate_eligibility(ineligible_user)

print(f"Status: {report2.status.value.upper()}")
print(f"Missing requirements:")
for req in report2.missing_requirements:
    print(f"  - {req}")

## 🤖 Agent 3: Conversation Manager

Orchestrates the full conversation flow between user and agents.

In [ ]:
# Create conversation manager
manager = ConversationManager()

print("=== Interactive Demo ===")
print("Simulating a conversation...\n")

# Simulate conversation inputs step by step
inputs = [
    "NSW",
    "80000",
    "700000",
    "yes",
    "citizen",
    "yes",
]

print(f"Bot: {manager.get_next_prompt()}")
for user_input in inputs:
    print(f"User: {user_input}")
    response = manager.process_input(user_input)
    print(f"Bot: {response}\n")

# Show final summary
summary = manager.get_summary()
print("="*50)
print(f"✅ Final Result: {summary['eligibility_status']}")
print(f"💰 Grant Amount: ${summary['grant_amount']:,}")

## 📄 Agent 4: Report Generator

Generates formatted eligibility reports.

In [ ]:
# Use the completed conversation data
user_profile = manager.user_profile.to_dict()
report = manager.eligibility_report

# Generate report
generator = ReportGenerator()
output_path = generator.generate_report(user_profile, report, output_format="markdown")

print(f"📄 Report saved to: {output_path}")

# Display report
print("\n" + "="*60)
print("GENERATED REPORT:")
print("="*60)
with open(output_path, "r") as f:
    print(f.read()[:2000])  # Show first 2000 chars

In [ ]:
# Also generate HTML report
html_path = generator.generate_report(user_profile, report, output_format="html")
print(f"\n📄 HTML report saved to: {html_path}")

## 📊 Batch Testing

Run multiple test cases to verify accuracy.

In [ ]:
import json

# Load sample test cases
with open("src/data/sample_users.json", "r") as f:
    test_cases = json.load(f)["test_cases"]

print(f"Running {len(test_cases)} test cases...\n")

passed = 0
failed = 0

for test in test_cases:
    user_data = test["user_data"]
    expected = test["expected"]
    
    # Get rules for this state
    rules = scraper.scrape_state_rules(user_data["state"])
    if not rules:
        print(f"⚠️  Skipping {test['id']}: No rules for {user_data['state']}")
        continue
    
    interpreter = RuleInterpreter(rules)
    result = interpreter.validate_eligibility(user_data)
    
    actual_eligible = result.status.value == "eligible"
    expected_eligible = expected["eligible"]
    
    if actual_eligible == expected_eligible:
        passed += 1
        status = "✅ PASS"
    else:
        failed += 1
        status = "❌ FAIL"
    
    print(f"{status} | {test['id']}: {test['description']}")
    if actual_eligible != expected_eligible:
        print(f"       Expected: {expected_eligible}, Got: {actual_eligible}")

print("\n" + "="*60)
print(f"📊 Test Results: {passed} passed, {failed} failed out of {len(test_cases)}")
print("="*60)

## 🎯 Interactive CLI (same as running `python cli.py`)

For a full interactive chat experience, run the CLI:

```
python src/chatbot/cli.py
```

Or check eligibility quickly:

```
python src/chatbot/cli.py check --state NSW --income 80000 --property-price 700000 --first-home
```

In [ ]:
print("\n🎉 Demo Complete!")
print("\nTo run the full Rasa chatbot, install Rasa and run:")
print("  rasa train")
print("  rasa run actions")
print("  rasa shell")

print("\nOr use the interactive CLI:")
print("  python src/chatbot/cli.py")